# Voight-Kampff Generative AI Authorship Verification 2024
* **Authors**: 
    - Pedro Amaya Moreno (pedro.amaya@alumnos.upm.es)
    - Alejandro Pardo Bascuñana (alejandro.pardo.bascunana@alumnos.upm.es)

Proposal for the [Voight-Kampff Generative AI Shared Task (PAN @ CLEF 2024)](https://pan.webis.de/clef24/pan24-web/generated-content-analysis.html), developed as part of the NLP course in the Master's program *Aprendizaje Automático y Datos Masivos* at Universidad Politécnica de Madrid (2024-2025).

The best model trained for 20 epochs is available at [HuggingFace](https://huggingface.co/Alejandro-Pardo/voight-kampff-pan2024-gte-en-v1.5)


In [ ]:
from sentence_transformers import SentenceTransformer, losses, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from transformers import EarlyStoppingCallback
from datasets import Dataset
from sentence_transformers.evaluation import TripletEvaluator
import numpy as np
np.random.seed(42)

## Proposed Design


Our proposed design is inspired by two specific teams from the competition. We also used paper [1], published after the competition, which discussed the problem formulations and how models were evaluated.

### `you-shun-you-de` [2]

They used a strategy called TSA (Triple Sentence Analysis) that consisted of splitting competition texts into smaller texts formed by three sentences.

These texts obtained with TSA were used to fine-tune a BERT model with a softmax activation layer to decide whether a sentence was AI-generated or not.
   
For model evaluation, they indicated that they performed the same splitting process on the full texts, classified these sub-texts, and computed a weighted average to obtain the classification of the complete text.
   
They did not indicate how they performed this weighting, but the interesting idea proposed by this solution is to divide the text classification task into a classification of smaller text chunks and use these to perform the complete text classification.

### `baselineavengers` [3]
This proposal is interesting because they achieved very good results without using a Transformer model. They used a `bag-of-words` to represent the texts and extracted term frequencies and a tf-idf for all tokens in the competition's training set.

They experimented with several classifiers (*Multinomial Naive Bayes*, *Logistic Regression*, and a *SVM (Support Vector Machine)* with a linear kernel) with the two features they extracted, and achieved very good results with the SVM using the tf-idf feature.

They also leveraged the problem formulation to use the following scoring function, where `t1` and `t2` are the two texts we want to classify:
$$P(human|t_1) = \frac{P(human|t_1) + (1 - P(LLM|t_2))}{2}$$

Where the results from both classifications are used to corroborate whether text t1 is human or not, using the average of the two classifications, which by definition should be:
$$P(human|t_1) = 1 - P(LLM|t_2)$$

### Our Proposed Design
First, we preprocess the texts and split them into three sets (`train`, `test`, and `validate`). The texts are divided into smaller chunks that we use to fine-tune a transformer embedding generation model.

The text `chunks` are designed to have a similar token size, unlike the first paper where they only ensured their small texts were formed by three sentences but did not indicate how they handled texts smaller than three sentences or when sentences were too large.

We also introduce noise to the text using `leetspeak`, which was something introduced by the competition organizers for the testing set. We also considered adding German texts to our training set, but ultimately decided to keep only English texts, as we did not have a significant amount of German AI-generated texts.

The embedding model used is `Alibaba-NLP/gte-base-en-v1.5`, a pre-trained English text embedding model developed by Alibaba's NLP team [5, 6]. We selected it based on the [MTEB Leaderboard](https://huggingface.co/spaces/mteb/leaderboard), given that its metrics were good and its memory footprint was not excessive for our development systems.

For fine-tuning, we use a contrastive learning technique, specifically through triplets as employed in [4]. This way, anchor texts (human) are pulled closer to other human texts and pushed away from AI texts in the vector space where embeddings are generated. For this, we use our validation dataset together with the `TripletEvaluator` class from `SentenceTransformer` and its corresponding `TripletLoss`.

Then, the fine-tuned embedding model is used to generate embeddings for the sentences in the training and validation sets to train a text classifier. For this, we use a Support Vector Machine with a linear kernel, since the embeddings already exhibit high linearity. Additionally, we use a calibrator to obtain prediction probabilities (since sklearn's `LinearSVM` class does not provide them by default), calibrating the model with the test dataset. This is used to return prediction confidence as in the original competition.

Once we have the `Classifier` to classify the embeddings of text `chunks`, to classify complete text pairs, we first split them into chunks of the same size as the training ones, compute the embeddings of both texts, and classify them with the `Classifier`. We obtain the average of the `chunk` probabilities for both texts and use them to calculate the probability of being human with the formula from the second paper mentioned above.

### References
- [1] A. A. Ayele et al., "Overview of PAN 2024: Multi-author Writing Style Analysis, Multilingual Text Detoxification, Oppositional Thinking Analysis, and Generative AI Authorship Verification Condensed Lab Overview", in Experimental IR Meets Multilinguality, Multimodality, and Interaction, 2024, pp. 231-259.
- [2] J. Huang, Y. Chen, M. Luo, and Y. Li, "Generative AI authorship verification of tri-sentence analysis base on the bert model", Working Notes of CLEF, 2024.
- [3] L. Lorenz, F. Z. Aygüler, F. Schlatt, and N. Mirzakhmedova, "BaselineAvengers at PAN 2024: often-forgotten baselines for LLM-generated text detection", Working Notes of CLEF, 2024.
- [4] H. Fabregat, D. Deniz, A. Duque, L. Araujo, and J. Martinez-Romo, "NLP-UNED at eRisk 2024: Approximate Nearest Neighbors with Encoding Refinement for Early Detecting Signs of Anorexia", in CEUR Workshop Proceedings, 2024, vol. 3740, pp. 813-824.
- [5] Xin Zhang, Yanzhao Zhang, Dingkun Long, Wen Xie, Ziqi Dai, Jialong Tang, Huan Lin, Baosong Yang, Pengjun Xie, Fei Huang, Meishan Zhang, Wenjie Li, & Min Zhang. (2024). mGTE: Generalized Long-Context Text Representation and Reranking Models for Multilingual Text Retrieval.
- [6] Zehan Li, Xin Zhang, Yanzhao Zhang, Dingkun Long, Pengjun Xie, & Meishan Zhang. (2023). Towards General Text Embeddings with Multi-stage Contrastive Learning.


## Data Augmentation with Ollama


As discussed in class, the results obtained by some models in the competition were not as expected. One of the reasons considered was the lack of diversity in the training data, since a different model was probably used for the final evaluation. To try to solve this problem, we proposed using Data Augmentation.

We generated new sentences by rewriting the originals with the help of the Ollama library and the LLMs `llama 3.2 1b`, `qwen 2.5 1b`, and `gemma 2 2b`. We also wanted to augment the data by creating German samples, translating data from the llama 3.2 model and human data with the helsinki-nlp/opus-mt-en-de model, but they were not included in the end, as we considered it would not be significant given the small amount of data.

The generated datasets are included in the `data/` directory.


In [ ]:
import ollama
import json

input_file = "data/human_parsed.jsonl"
output_file = "gemma2_2b.jsonl" #qwen2.5:1b...
index_file = "last.txt"
model = "gemma2:2b"

ollama.pull(model)

def rewrite_text(texto):
    prompt = f"The following is a human-written text. Now, please rewrite this text in your writing style, also optimize sentence structures and correct grammatical errors. Answer only with the rewritten text: \n {texto}"
    # The following is a human-written text. Now, please rewrite this text in your writing style and then translate the text to German. Only answer with the rewritten German text: \n {texto}
    response = ollama.generate(model=model, prompt=prompt)
    # Captura la respuesta generada
    rewritten_text = response["response"]
    return rewritten_text

def obtain_last_index():
    try:
        with open(index_file, "r") as f:
            return int(f.read().strip())
    except:
        #creaye the file if it does not exist
        with open(index_file, "w") as f:
            f.write("-1")

def save_last_index(indice):
    # Guardar el último índice procesado
    with open(index_file, "w") as f:
        f.write(str(indice))

In [ ]:
last_index = obtain_last_index()

with open(input_file, "r", encoding="utf-8") as infile, open(output_file, "a", encoding="utf-8") as outfile: 
    for idx, line in enumerate(infile): 
        if idx < last_index:
            continue  # Skip lines until reaching the last processed index
        
        data = json.loads(line)
        rewritten_text = rewrite_text(data["text"])
        
        # Create new entry with ID in required format and label 1 
        new_entry = { 
            "id": data["id"], 
            "text": rewritten_text, 
            "label": 1 
        }
        outfile.write(json.dumps(new_entry, ensure_ascii=False) + "\n")

        # Guardar el índice actual como el último procesado
        save_last_index(idx + 1)

In [ ]:
last_index = obtain_last_index()

with open(input_file, "r", encoding="utf-8") as infile, open(output_file, "a", encoding="utf-8") as outfile: 
    for idx, line in enumerate(infile): 
        if idx < last_index:
            continue  # Skip lines until reaching the last processed index
        
        data = json.loads(line)
        rewritten_text = rewrite_text(data["text"])
        
        # Create new entry with ID in required format and label 1 
        new_entry = { 
            "id": data["id"], 
            "text": rewritten_text, 
            "label": 1 
        }
        outfile.write(json.dumps(new_entry, ensure_ascii=False) + "\n")

        # Guardar el índice actual como el último procesado
        save_last_index(idx + 1)

For text translation, we used the `transformers` library from HuggingFace, which allows us to use pipelines and pre-trained text translation models. We used the `Helsinki-NLP/opus-mt-en-de` model to translate texts from English to German.


In [ ]:
from transformers import pipeline, AutoTokenizer
import json

input_file = "data/machines/llama32_1b.jsonl" #data/human_parsed.jsonl
output_file = "llama32_helsinki-nlp_german.jsonl" #data/human_german.jsonl
index_file = "last2.txt"
model = "Helsinki-NLP/opus-mt-en-de"
tokenizer = AutoTokenizer.from_pretrained(model)

# Load the translation pipeline
translator = pipeline("translation_en_to_de", model=model, tokenizer=tokenizer, device=0)

In [ ]:
def translate_text(texto):
    # Translate text to German using the model
    translated_text = translator(texto, max_length=512, truncation = True)[0]['translation_text']
    return translated_text

last_index = obtain_last_index()

with open(input_file, "r", encoding="utf-8") as infile, open(output_file, "a", encoding="utf-8") as outfile: 
    for idx, line in enumerate(infile): 
        if idx < last_index:
            continue  # Skip lines until reaching the last processed index
        
        data = json.loads(line)
        translated_text = translate_text(data["text"])
        
        # Create new entry with ID in required format and label 1 
        new_entry = { 
            "id": data["id"], 
            "text": translated_text, 
            "label": 1 
        }
        outfile.write(json.dumps(new_entry, ensure_ascii=False) + "\n")

        # Guardar el índice actual como el último procesado
        save_last_index(idx + 1)

In [ ]:
input_file = "data/human_parsed.jsonl" #data/human_parsed.jsonl
output_file = "data/human_german.jsonl" #data/human_german.jsonl
index_file = "last.txt"

last_index = obtain_last_index()

with open(input_file, "r", encoding="utf-8") as infile, open(output_file, "a", encoding="utf-8") as outfile: 
    for idx, line in enumerate(infile): 
        if idx < last_index:
            continue  # Skip lines until reaching the last processed index
        
        data = json.loads(line)
        translated_text = translated_text(data["text"])
        
        # Create new entry with ID in required format and label 1 
        new_entry = { 
            "id": data["id"], 
            "text": translated_text, 
            "label": 0 
        }
        outfile.write(json.dumps(new_entry, ensure_ascii=False) + "\n")

        # Guardar el índice actual como el último procesado
        save_last_index(idx + 1)

Finally, to aggregate the generated data and return the dataset to its original state, the sentences were joined by id and a new file was created with the augmented dataset appended.


In [ ]:
import json
from collections import defaultdict

input_file = ['data/machines/gemma2_2b.jsonl', 'data/machines/llama32_1b.jsonl', 'data/machines/qwen25_1b.jsonl', 'data/machines/llama32_helsinki-nlp_german.jsonl', 'data/human_german.jsonl']
output_file = ['gemma2_2b_combined.jsonl', 'llama32_1b_combined.jsonl', 'qwen25_1b_combined.jsonl', 'llama32_helsinki-nlp_german_combined.jsonl', 'human_german_combined.jsonl']

# Dictionary to hold aggregated texts by article ID
articles = defaultdict(lambda: {"text": "", "label": None})

# Read the input JSONL file and aggregate texts by article
for in_file, out_file in zip(input_file, output_file):
    with open(in_file, "r", encoding="utf-8") as infile:
        for line in infile:
            data = json.loads(line)
            article_id = data["id"].rsplit('-', 1)[0]
            articles[article_id]["text"] += " " + data["text"]
            articles[article_id]["label"] = data["label"]

    # Write the aggregated texts to the output JSONL file
    with open(out_file, "w", encoding="utf-8") as outfile:
        for article_id, content in articles.items():
            out_data = {"id": article_id, "text": content["text"].strip(), "label": content["label"]}
            outfile.write(json.dumps(out_data, ensure_ascii=False) + "\n")

print("Aggregation complete. Check the output JSONL files.")

We ended up not using the German texts because we were not confident in the quality of the texts we had obtained, since the texts translated to German might be considered as AI-generated (as translation models probably also introduce patterns that differentiate human text from AI text).


## Data Preprocessing


First, we downloaded the texts from the competition website [PAN](https://pan.webis.de/clef24/pan24-web/generated-content-analysis.html). We stored them locally in a `/data` folder, with AI texts in directories inside `/data/machines` in folders named after the model and the texts in `.jsonl` documents within them. Human text documents are stored in the root of `/data`.

To process the data we use the `DatasetLoader` class, which handles obtaining text chunks, splitting them into train, test, and validate sets, and creating dataset generators for training.

For splitting texts into chunks, we add to the dictionaries representing documents a value called `tk_positions` that represents the text positions of `K`-token chunks (we use chunks of 32 tokens with a tolerance of 2, meaning text chunks whose tokens are between [16, 48] tokens (~64-~192 words)). We use the `Alibaba-NLP/gte-base-en-v1.5` tokenizer since that is the one we will use for fine-tuning the model.

To split them into the mentioned sets, we save them in directories named after `dataset_types` with the specified distribution. This information is stored in `loader_info.json` which contains the following:

```json
{
    "token_size": 32,
    "tolerance": 2,
    "tokenizer_name": "Alibaba-NLP/gte-base-en-v1.5",
    "files_per_document": 4000,
    "file_format": "processed_files_%d.jsonl",
    "dataset_info": {
        "ai-storage-name": "ai-docs",
        "human-storage-name": "human-docs",
        "document-info-name": "document_info.json"},
    "dataset_types": {
        "train": 0.75,
        "validate": 0.15,
        "test": 0.1
    }
}
```
With this information, we can reload it without any issues in subsequent runs. Additionally, we also save for each document silo the `document_info.json` file which contains:

```json
{
    "ai-docs": {
        "location": ".\\documents\\train\\ai-docs",
        "file_lengths": [11, 4, 9, 5, 11...],
        "last_file":".\\documents\\train\\ai-docs\\processed_files_40000.jsonl"
    },
    "human-docs": {
        "location": ".\\documents\\train\\human-docs",
        "file_lengths": [25, 23, 14, 13...],
        "last_file": ".\\documents\\train\\human-docs\\processed_files_0.jsonl"
    }
}
```
Where `file_lengths` are the lengths in token chunks of the documents stored in the silo (train, test, validate), and the location of each part.



### Utility Functions, DatasetLoader, and DatasetGenerator Code


The code includes functions for reading and saving documents (both jsonl and saving Python dictionaries in JSON format).

Since we were working on both Windows and Mac, we looked for a way to use paths from both Windows and Mac without compatibility issues. To solve this problem, we used the `AgnosticPath` class found in a StackOverflow post that perfectly suited our needs.

The `DatasetLoader` class handles processing documents (to obtain their token positions) and saving them into the desired sets with the desired distributions. The `DatasetGenerator` class is used to generate random datasets based on the data stored in the `DatasetLoader` for training, validation, and testing of our models.


In [11]:
import os
import re
from pathlib import Path, PureWindowsPath
from dataclasses import dataclass
from sklearn.model_selection import train_test_split;
from typing import List, Tuple, Dict, Any
import json
from transformers import AutoTokenizer
from copy import copy, deepcopy
import numpy.random as rng
from itertools import chain
from random import sample
from pyleetspeak import LeetSpeaker

def write_document_file(file_path:str, file_list:List[dict]):
    with open(file_path, mode='w', encoding='utf-8') as fd:
        fd.writelines([f"{json.dumps(file_info)}\n" for file_info in file_list])


def load_document_file(file_path:str):
    with open(file_path, mode='r', encoding='utf-8') as fd:
        lines = fd.readlines()
    return [json.loads(line) for line in lines]


def save_file(file_path:str, file: dict):
    with open(file_path, mode='w', encoding='utf-8') as fd:
        json.dump(file, fd)


def load_file(file_path:str):
    with open(file_path, mode='r', encoding='utf-8') as fd:
        file_dict = json.load(fd)
    return file_dict

class AgnosticPath(Path): # From https://stackoverflow.com/questions/60291545/converting-windows-path-to-linux
    """A class that can handle input with Windows (\\) and/or posix (/) separators for paths"""

    def __new__(cls, *args, **kwargs):
        new_path = PureWindowsPath(*args).parts
        if (os.name != "nt") and (len(new_path) > 0) and (new_path[0] in ("/", "\\")):
          new_path = ("/", *new_path[1:])
        return super().__new__(Path, *new_path, **kwargs)

class DatasetLoader:
    """
    You can modify the storage_information attributes updating the following dict (only works on initial load) passed
    on the init argument:
    storage_information = {
        "token_size": 32,
        "tolerance": 2,
        "tokenizer_name": "Alibaba-NLP/gte-base-en-v1.5",
        "files_per_document": 4000,
        "file_format": "processed_files_%d.jsonl",
        "dataset_info": {
            "ai-storage-name": "ai-docs",
            "human-storage-name": "human-docs",
            "document-info": "document_info.json"
        },
        "dataset_types": {"train": 0.6, "validate": 0.2, "test": 0.2}
    }
    """
    def __init__(self,
                 storage_location:str="documents",
                 storage_information_load_name:str="loader_info.json",
                 storage_information: Dict[str, Any] = None):
        self.storage_location = AgnosticPath(".") / storage_location
        # Load storage information
        storage_information_path = self.storage_location / storage_information_load_name
        if self.storage_location.exists():
            st_information = load_file(storage_information_path)
        else:
            st_information = {
                "token_size": 32,
                "tolerance": 2,
                "tokenizer_name": "Alibaba-NLP/gte-base-en-v1.5",
                "files_per_document": 4000,
                "file_format": "processed_files_%d.jsonl",
                "dataset_info": {
                    "ai-storage-name": "ai-docs",
                    "human-storage-name": "human-docs",
                    "document-info-name": "document_info.json"
                },
                "dataset_types": {"train": 0.6, "validate": 0.2, "test": 0.2}
            }
            if storage_information:
                st_information.update(storage_information)
            assert (not any(v < 0. for v in st_information["dataset_types"].values()) and
                    sum(st_information["dataset_types"].values()) == 1.), "Invalid sizes for the dataset types"
            os.mkdir(self.storage_location)
            save_file(storage_information_path, st_information)
        # Loaded parameters
        self.token_size = st_information["token_size"]
        self.tolerance = st_information["tolerance"]
        self.tokenizer = AutoTokenizer.from_pretrained(st_information["tokenizer_name"])
        self.files_per_document = st_information["files_per_document"]
        self.file_format = st_information["file_format"]
        self.dataset_info = st_information["dataset_info"]
        self.dataset_types = st_information["dataset_types"]
        # Generated setup
        for dataset_name in self.dataset_types:
            self.setup(dataset_name)
    
    def setup(self, dataset_name: str):
        memory_path = self.storage_location / dataset_name
        ai_path = memory_path / self.dataset_info["ai-storage-name"]
        human_path = memory_path / self.dataset_info["human-storage-name"]
        document_info_path = memory_path / self.dataset_info["document-info-name"]
        # Create directories
        if all(path.exists() for path in (memory_path, ai_path, human_path)):
            return
        if not memory_path.exists():
            os.mkdir(str(memory_path))
        if not ai_path.exists():
            os.mkdir(str(ai_path))
        if not human_path.exists():
            os.mkdir(str(human_path))
        first_ai_file = str(ai_path / (self.file_format % (0,)))
        first_human_file = str(human_path / (self.file_format % (0,)))
        write_document_file(first_ai_file, [])
        write_document_file(first_human_file, [])
        document_info = {
            "ai-docs": {
                "location": str(ai_path),
                "file_lengths": [],
                "last_file": first_ai_file,
            },
            "human-docs": {
                "location": str(human_path),
                "file_lengths": [],
                "last_file": first_human_file,
            }
        }
        save_file(document_info_path, document_info)
    
    def preprocess_text(self, text: dict):
        """
        Processes the passed text and obtains the token chunks to save with the document.
        """
        variance = int(self.token_size / self.tolerance)
        tk_variance = (self.token_size - variance, self.token_size + variance)
        text_positions = [0]
        last_position = 0
        prev_token = 0
        for phrase in re.split(r"[.!?\n\r]", text["text"]):
            if len(phrase) == 0:
                last_position += 1
                continue
            curr_position = last_position + len(phrase) + 1
            curr_tokens = len(self.tokenizer.tokenize(text["text"][text_positions[-1]:curr_position]))
            if prev_token > tk_variance[0] and curr_tokens > tk_variance[1]:
                text_positions.append(last_position)
                prev_token = curr_tokens - prev_token
            else:
                prev_token = curr_tokens
            last_position += len(phrase) + 1
        if prev_token != 0:
            text_positions.append(last_position)
        text["tk_positions"] = text_positions

    def save_file_info(self, text:dict, files:List[dict], file_info:dict):
        """
        Saves the processed text into files (modifiyng the corresponding file_info) and if files > files_per_document,
        saves the current document and opens a new one.
        """
        files.append(text)
        file_info["file_lengths"].append(len(text["tk_positions"]))
        # If maximum files per document are reached, save the current files and open a new one
        if len(files) % self.files_per_document == 0:
            write_document_file(file_info["last_file"], files)
            files.clear()
            file_info["last_file"] = str(AgnosticPath(file_info["location"]) / (self.file_format % (len(file_info["file_lengths"]),)))

    def preprocess_dataset_files(self, dataset_location:str, texts_to_load: List[dict]):
        """
        Preprocesses dataset files that are meant to be loaded to the preprocessor / document generator. The files to
        load must be dictionaries such that they have a `text` and a `label` (0 human / 1 AI). 
        """
        document_info_path = str(AgnosticPath(dataset_location) / self.dataset_info["document-info-name"])
        document_info = load_file(document_info_path)
        ai_info = document_info["ai-docs"]
        human_info = document_info["human-docs"]
        # Load Last document files
        ai_files = load_document_file(ai_info["last_file"])
        human_files = load_document_file(human_info["last_file"])
        for text in texts_to_load:
            self.preprocess_text(text)
            if text["label"] == 0:
                self.save_file_info(text, human_files, human_info)
            else:
                self.save_file_info(text, ai_files, ai_info)
        write_document_file(human_info["last_file"], human_files)
        write_document_file(ai_info["last_file"], ai_files)
        save_file(document_info_path, document_info)
    
    def preprocess_files(self, texts_to_load:List[dict]):
        """
        Preprocesses files that are meant to be loaded to the preprocessor / document generator. The files to load must be
        dictionaries such that they have a `text` and a `label` (0 human / 1 AI). 
        """
        curr_total = 1.
        curr_texts = texts_to_load
        num_datasets = len(self.dataset_types)
        for idx, dataset_name in enumerate(self.dataset_types):
            dataset_path = str(self.storage_location / dataset_name)
            if idx == num_datasets - 1:
                self.preprocess_dataset_files(dataset_path, curr_texts)
                continue
            train_size = self.dataset_types[dataset_name] / curr_total
            files_to_add, curr_texts = train_test_split(curr_texts, train_size=train_size)
            self.preprocess_dataset_files(dataset_path, files_to_add)
            curr_total -= self.dataset_types[dataset_name]
    
    def create_generators(self,
                          n_test_cases: int,
                          token_size:int=2,
                          rep_noisy_cases:float=1.0,
                          change_prob:float=0.8,
                          change_freq:float=0.8
                          ):
        return {dataset_name: DatasetGenerator(
            document_info_path=str(self.storage_location / dataset_name / self.dataset_info["document-info-name"]),
            files_per_document=self.files_per_document,
            file_format=self.file_format,
            n_test_cases=n_test_cases,
            token_size=token_size,
            rep_noisy_cases=rep_noisy_cases,
            change_prob=change_prob,
            change_freq=change_freq
        ) for dataset_name in self.dataset_types}

@dataclass
class DatasetGenerator:
    document_info_path: str
    files_per_document: int
    file_format: str
    n_test_cases: int
    token_size:int=2,
    rep_noisy_cases:float=1.0,
    change_prob:float=0.8,
    change_freq:float=0.8

    def __post_init__(self):
        self.document_info = load_file(self.document_info_path)
    
    def _rng_text_chunk(self, file_info: List[int], size:int):
        return [
            (text_id, rng.randint(0, file_info[text_id] - self.token_size) if file_info[text_id] > self.token_size else 0)
            for text_id in rng.randint(0, len(file_info), size=size)
        ]
    
    def _read_text_chunks(self, doc_type:str, text_list_id: List[Tuple[int, int]]):
        files_per_document = self.files_per_document
        file_path = str(AgnosticPath(self.document_info[doc_type]["location"]) / self.file_format)
        text_dict = {}
        file_start = 0
        texts = load_document_file(file_path % (file_start,))
        for doc_id, pos in text_list_id:
            while doc_id - file_start >= files_per_document:
                file_start += files_per_document
                texts = load_document_file(file_path % (file_start,))
            text = texts[doc_id - file_start]
            tk_positions = text["tk_positions"]
            if pos + self.token_size >= len(tk_positions):
                text_dict[(doc_id, pos)] = text["text"][tk_positions[pos]:]
            else:
                text_dict[(doc_id, pos)] = text["text"][tk_positions[pos]:tk_positions[pos + self.token_size]]
        return text_dict
    
    def _rng_text(self, file_info: List[int], size:int):
        return rng.randint(0, len(file_info), size=size)
    
    def _read_text(self, doc_type:str, text_ids: List[int]):
        files_per_document = self.files_per_document
        file_path = str(AgnosticPath(self.document_info[doc_type]["location"]) / self.file_format)
        text_dict = {}
        file_start = 0
        texts = load_document_file(file_path % (file_start,))
        for doc_id in text_ids:
            while doc_id - file_start >= files_per_document:
                file_start += files_per_document
                texts = load_document_file(file_path % (file_start,))
            text = texts[doc_id - file_start]
            text_dict[doc_id] = text["text"] # {"id": text["id"], "text": text["text"]}
        return text_dict

    def _generate_test_cases_tuples(self, min_dist:int=0, max_dist:int=1, only_ai_human:bool=False):
        ai_info = self.document_info["ai-docs"]
        human_info = self.document_info["human-docs"]
        if only_ai_human:
            num_cases = int(self.n_test_cases // 2)
            # Text types
            human_ai = (self._rng_text_chunk(human_info["file_lengths"], num_cases), self._rng_text_chunk(ai_info["file_lengths"], num_cases))
            ai_human = (self._rng_text_chunk(ai_info["file_lengths"], num_cases), self._rng_text_chunk(human_info["file_lengths"], num_cases))
            # Texts to read
            human_dict = self._read_text_chunks("human-docs", sorted(chain(human_ai[0], ai_human[1])))
            ai_dict = self._read_text_chunks("ai-docs", sorted(chain(human_ai[1], ai_human[0])))
            return [
                *((human_dict[h], ai_dict[ai], max_dist) for h, ai in zip(*human_ai)),
                *((ai_dict[ai], human_dict[h], max_dist) for ai, h in zip(*ai_human)),
            ]
        num_cases = int(self.n_test_cases // 4)
        # Text types
        human_human = (self._rng_text_chunk(human_info["file_lengths"], num_cases), self._rng_text_chunk(human_info["file_lengths"], num_cases))
        human_ai = (self._rng_text_chunk(human_info["file_lengths"], num_cases), self._rng_text_chunk(ai_info["file_lengths"], num_cases))
        ai_human = (self._rng_text_chunk(ai_info["file_lengths"], num_cases), self._rng_text_chunk(human_info["file_lengths"], num_cases))
        ai_ai = (self._rng_text_chunk(ai_info["file_lengths"], num_cases), self._rng_text_chunk(ai_info["file_lengths"], num_cases))
        # Texts to read
        human_dict = self._read_text_chunks("human-docs", sorted(chain(human_human[0], human_human[1], human_ai[0], ai_human[1])))
        ai_dict = self._read_text_chunks("ai-docs", sorted(chain(ai_ai[0], ai_ai[1], human_ai[1], ai_human[0])))
        return [
            *((human_dict[h1], human_dict[h2], min_dist) for h1, h2 in zip(*human_human)),
            *((human_dict[h], ai_dict[ai], max_dist) for h, ai in zip(*human_ai)),
            *((ai_dict[ai], human_dict[h], max_dist) for ai, h in zip(*ai_human)),
            *((ai_dict[ai1], ai_dict[ai2], min_dist) for ai1, ai2 in zip(*ai_ai))
        ]
    
    def _add_noise(self, text:str):
        leet_speaker = LeetSpeaker(text, change_prb=self.change_prob, change_frq=self.change_freq, mode="basic")
        return leet_speaker.text2leet()

    def generate_tuples(self, min_dist:int=0, max_dist:int=1, only_ai_human:bool=False):
        test_cases = self._generate_test_cases_tuples(min_dist, max_dist, only_ai_human)
        if not self.rep_noisy_cases == 0.:
            noisify_cases = (copy(test_cases) if self.rep_noisy_cases == 1. else
                             sample(test_cases, int(len(test_cases)*self.rep_noisy_cases)))
            noisy_cases = []
            for t1, t2, label in noisify_cases:
                rng_values = rng.randint(0, 2, size = 2)
                t1_noisy = self._add_noise(t1) if 0 in rng_values else t1
                t2_noisy = self._add_noise(t2) if 1 in rng_values else t2
                noisy_cases.append((t1_noisy, t2_noisy, label))
            test_cases.extend(noisy_cases)
        return test_cases
    
    def _generate_test_cases_singles(self, human_label:int=0, ai_label:int=1):
        num_cases = int(self.n_test_cases // 2)
        human_l = self._rng_text_chunk(self.document_info["human-docs"]["file_lengths"], num_cases)
        ai_l = self._rng_text_chunk(self.document_info["ai-docs"]["file_lengths"], num_cases)
        human_dict = self._read_text_chunks("human-docs", sorted(human_l))
        ai_dict = self._read_text_chunks("ai-docs", sorted(ai_l))
        return [
            *((human_dict[h], human_label) for h in human_l),
            *((ai_dict[ai], ai_label) for ai in ai_l),
        ]

    def generate_singles(self, human_label:int=0, ai_label:int=1):
        test_cases = self._generate_test_cases_singles(human_label, ai_label)
        if not self.rep_noisy_cases == 0.:
            noisify_cases = (copy(test_cases) if self.rep_noisy_cases == 1. else
                             sample(test_cases, int(len(test_cases)*self.rep_noisy_cases)))
            noisy_cases = ((self._add_noise(t), v) for t, v in noisify_cases)
            test_cases.extend(noisy_cases)
        return test_cases
    
    def _generate_test_cases_triplets(self, same_positive_anchor:bool=False):
        anchor = self._rng_text_chunk(self.document_info["human-docs"]["file_lengths"], self.n_test_cases)
        positive = (anchor if same_positive_anchor else
                    self._rng_text_chunk(self.document_info["human-docs"]["file_lengths"], self.n_test_cases))
        negative = self._rng_text_chunk(self.document_info["ai-docs"]["file_lengths"], self.n_test_cases)
        human_dict = self._read_text_chunks("human-docs", sorted(chain(anchor, positive)))
        ai_dict = self._read_text_chunks("ai-docs", sorted(negative))
        return {
            "anchor": [human_dict[v] for v in anchor],
            "positive": [human_dict[v] for v in positive],
            "negative": [ai_dict[v] for v in negative]
        }

    def generate_triplets(self, same_positive_anchor:bool=False):
        triplet_cases = self._generate_test_cases_triplets(same_positive_anchor)
        if not self.rep_noisy_cases == 0.:
            for test_cases in triplet_cases.values():
                noisify_cases = (copy(test_cases) if self.rep_noisy_cases == 1. else
                                sample(test_cases, int(len(test_cases)*self.rep_noisy_cases)))
                test_cases.extend(map(self._add_noise, noisify_cases))
        return triplet_cases

    def _generate_random_pairings(self, is_human:float=0.):
        ai_info = self.document_info["ai-docs"]
        human_info = self.document_info["human-docs"]
        num_cases = int(self.n_test_cases // 2)
        human_ai = (self._rng_text(human_info["file_lengths"], num_cases), self._rng_text(ai_info["file_lengths"], num_cases))
        ai_human = (self._rng_text(ai_info["file_lengths"], num_cases), self._rng_text(human_info["file_lengths"], num_cases))
        human_dict = self._read_text("human-docs", sorted(chain(human_ai[0], ai_human[1])))
        ai_dict = self._read_text("ai-docs", sorted(chain(human_ai[1], ai_human[0])))
        return [
            *({"text1": human_dict[h], "text2": ai_dict[ai], "is_human": is_human} for h, ai in zip(*human_ai)),
            *({"text1": ai_dict[ai], "text2": human_dict[h], "is_human": 1. - is_human} for ai, h in zip(*ai_human)),
        ]

    def generate_random_pairings(self, is_human:float=0.):
        random_pairings = self._generate_random_pairings(is_human)
        if not self.rep_noisy_cases == 0.:
            noisify_cases = (deepcopy(random_pairings) if self.rep_noisy_cases == 1. else
                             sample(random_pairings, int(len(random_pairings)*self.rep_noisy_cases)))
            noisy_cases = []
            for texts in noisify_cases:
                rng_values = rng.randint(0, 2, size = 2)
                noisy_texts = deepcopy(texts)
                if 0 in rng_values:
                    noisy_texts["text1"] = self._add_noise(noisy_texts["text1"])
                if 1 in rng_values:
                    noisy_texts["text2"] = self._add_noise(noisy_texts["text2"])
                noisy_cases.append(noisy_texts)
            random_pairings.extend(noisy_cases)
        return random_pairings


### Data Loading and Preprocessing


In [ ]:
import os

loader = DatasetLoader(
    storage_location="documents",
    storage_information={
        "tokenizer_name": "Alibaba-NLP/gte-base-en-v1.5",
        "dataset_types": {"train": 0.75, "validate": 0.15, "test": 0.1}
    }
)

In this cell we load the documents into the `DatasetLoader`, excluding the German files for the reasons mentioned previously.


In [ ]:
ai_data_path = os.path.join(".", "data", "machines")
ai_origins = [
    "alpaca-7b",
    "gpt-4-turbo-preview",
    "mistralai-mixtral-8x7b-instruct-v0.1",
    "bigscience-bloomz-7b1",
    "llama32_1b",
    "qwen-qwen1.5-72b-chat-8bit",
    "chavinlo-alpaca-13b",
    # "llama32_helsinki-nlp_german",
    "qwen25_1b",
    "gemini-pro",
    "meta-llama-llama-2-70b-chat-hf",
    "text-bison-002",
    "gemma2_2b",
    "meta-llama-llama-2-7b-chat-hf",
    "vicgalle-gpt2-open-instruct-v1",
    "gpt-3.5-turbo-0125",
    "mistralai-mistral-7b-instruct-v0.2",   
]

human_data_path = os.path.join(".", "data")
human_origins = [
    "human",
    # "human_german_combined"
]

for origin in ai_origins:
    print(f"Loading AI {origin=}")
    file_path = os.path.join(ai_data_path, origin, f'{origin}.jsonl')
    texts = load_document_file(file_path)
    for text in texts:
        text["origin"] = origin
        text["label"] = 1
    loader.preprocess_files(texts)

for origin in human_origins:
    print(f"Loading Human {origin=}")
    file_path = os.path.join(human_data_path, f'{origin}.jsonl')
    texts = load_document_file(file_path)
    for text in texts:
        text["origin"] = "human_text_competition"
        text["label"] = 0
    loader.preprocess_files(texts)

Loading AI origin='alpaca-7b'
Loading AI origin='gpt-4-turbo-preview'
Loading AI origin='mistralai-mixtral-8x7b-instruct-v0.1'
Loading AI origin='bigscience-bloomz-7b1'
Loading AI origin='llama32_1b'
Loading AI origin='qwen-qwen1.5-72b-chat-8bit'
Loading AI origin='chavinlo-alpaca-13b'
Loading AI origin='qwen25_1b'
Loading AI origin='gemini-pro'
Loading AI origin='meta-llama-llama-2-70b-chat-hf'
Loading AI origin='text-bison-002'
Loading AI origin='gemma2_2b'
Loading AI origin='meta-llama-llama-2-7b-chat-hf'
Loading AI origin='vicgalle-gpt2-open-instruct-v1'
Loading AI origin='gpt-3.5-turbo-0125'
Loading AI origin='mistralai-mistral-7b-instruct-v0.2'
Loading Human origin='human'


Finally, we created a function called `create_generators` that creates our custom dataset generators `DatasetGenerators`, which can generate text tuples (AI-Human, Human-AI, Human-Human, AI-AI), triplets (anchor, positive, negative), singles (Human - 0, AI - 1), and random text pairings (like tuples, but instead of text chunks, we use complete texts). To introduce noise to the sentences, we use the `pyleetspeak` package to introduce LeetSpeak into the text (e.g., h0l4 q_3 t4l).

In `create_generators` we can specify the following arguments:
* **n_test_cases: int** -> Number of cases we want the functions to generate.
* **token_size: int** -> Size of the text chunks (e.g., size 1 would be chunks of ~32 tokens, and in our case size 2 would be ~64 tokens).
* **rep_noisy_cases: float** -> Percentage of cases that we copy from the generated cases to introduce noise. This means that if we want to generate 1000 cases with 0.15 noise, we will get 1150 cases with 150 noisy cases.
* **change_prob: float** -> Probability of introducing Leet Speak noise to a word.
* **change_freq: float** -> Amount of changes produced by Leet Speak in the word.



In [5]:
generators = loader.create_generators(
    n_test_cases=10000,
    rep_noisy_cases=0.15,
    change_prob=0.2,
    change_freq=0.2
)
for gen_name in generators:
    print(f'{gen_name}: {generators[gen_name]}')

train: DatasetGenerator(document_info_path='documents\\train\\document_info.json', files_per_document=4000, file_format='processed_files_%d.jsonl', n_test_cases=10000, token_size=2, rep_noisy_cases=0.15, change_prob=0.2, change_freq=0.2)
validate: DatasetGenerator(document_info_path='documents\\validate\\document_info.json', files_per_document=4000, file_format='processed_files_%d.jsonl', n_test_cases=10000, token_size=2, rep_noisy_cases=0.15, change_prob=0.2, change_freq=0.2)
test: DatasetGenerator(document_info_path='documents\\test\\document_info.json', files_per_document=4000, file_format='processed_files_%d.jsonl', n_test_cases=10000, token_size=2, rep_noisy_cases=0.15, change_prob=0.2, change_freq=0.2)


In [5]:
data = generators['train'].generate_tuples()
# Creates a tuple text1, text2, distance (0 same (AI-AI, human-human), 1 different (AI-Human))
print(data[-1])
print(len(data))

("The New York Times first reported that a grand jury was hearing testimony against Chauvin. The Justice Department declined to c0mment.\n\nJury selection begins in Chavin's state case on March 8 -- he faces sec0nd-degree murder and manslaughter charges -- with opening statements scheduled for March 29.\n\n", '"Over and over again Facebook chooses expansion in new areas over sticking the landing on what they\'ve already done," Haugen said. "And I find it unconscionable that, as you read through the documents, it states very clearly there needs to be more resources on very basic safety systems.', 0)
11500


In [6]:
data = generators['train'].generate_singles()
# Creates a tuple text, class (0 human, 1 AI)
print(data[-1])

("With Trump's time in office set to expire at noon on January 20, House Speaker Nancy Pelosi @lso gave Trump and Vice President Mike Pence the option of @voiding impeachment if either Trump resigned or Pence mobilized the Cabinet to use the 25th Amendment to remove him from office.\n\nWhen those two offramps were ignored, Democrats in the House moved quickly toward impeachment and the first post-presidenti@l impeachment tri@l in US history.\n\n", 0)


In [7]:
data = generators['train'].generate_triplets()
for k, v in data.items():
    print(f"------ {k} ------ | NOT NOISY\n{v[0]}\n")
    print(f"------ {k} ------ | NOISY\n{v[-1]}\n")


------ anchor ------ | NOT NOISY
Don Lemon: Honestly, Chris, we're fearless in those moments because we are really vulnerable. And I think you always have my back and I know I always have yours. But that doesn't mean that it's going to be an agree-fest. Am I wrong?

Chris Cuomo: I think that's 100 percent accurate. What's the blessing?

------ anchor ------ | NOISY
Ivanka Trump and Jared Kushner Are Reportedly Already Unwelcome at Their Billionaire's Bunker Country Club

Ivanka Trump and Jared Kushner have only been landowners in Miami's exclusive "Billionaire's Bunker" enclave since December, but it seems the couple is already having some issues with their new neighbors.



------ positive ------ | NOT NOISY
 And in a statement this week, Twitter would not commit to banning those representing Islamic extremist governments from using its powerful platform.

"The situation in Afghanistan is rapidly evolving," said a Twitter spokesperson. "We're also witnessing people in the country usin

In [6]:
data = generators['train'].generate_random_pairings()
print(data[0])
print(data[-1])

{'text1': 'Tiger Woods responds to golfers wearing his Sunday red as he recovers from accident\n\nIt looks like Tiger Woods is watching some golf on TV while he recovers from last week\'s car accident in Los Angeles.\n\nDuring Sunday\'s final rounds, players in the World Golf Championships-Workday Championship, Puerto Rico Open and Gainbridge LPGA showed their support for Woods in various ways.\n\nRory McIlroy, Jason Day, Justin Thomas, Tony Finau, Patrick Reed, Tommy Fleetwood, Scottie Scheffler, Carlos Ortiz, and Cameron Champ rocked similar versions of his signature Sunday-red shirt and black pants. Billy Horschel had "TW" etched on his hat while Matt Kuchar, Jason Day and Bryson DeChambeau played with golf balls stamped with "TIGER."\n\nGolfers on supporting Tiger\n\nRory McIlroy: "It\'s just a gesture to let him know that we\'re thinking about him and we\'re rooting for him. Obviously things are looking a little better today than they were on Tuesday, but he\'s still got aways to 

In [6]:
train = generators["train"].generate_triplets()
train = Dataset.from_dict(train)
evaluate = generators["validate"].generate_triplets()
evaluate = Dataset.from_dict(evaluate)

The processed documents have been stored in the `documents/` directory to make our results reproducible.


## Training the Embedding Generation Model


The embedding model used is Alibaba-NLP/gte-base-en-v1.5, a pre-trained English text embedding model. We perform contrastive learning with triplets using our triplet training dataset, and fine-tune it with the validation dataset.


In [31]:
model = SentenceTransformer('Alibaba-NLP/gte-base-en-v1.5', trust_remote_code=True, device="cuda")
print(model)

SentenceTransformer(
  (0): Transformer({'max_seq_length': 8192, 'do_lower_case': False}) with Transformer model: NewModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)


We also freeze 10 of the base model's 12 layers, so that the weights of these layers are not updated and the pre-trained knowledge is preserved. We opted not to add a softmax layer, as we chose to use a Support Vector Machine classifier with a linear kernel, since the embeddings already exhibit high linearity.


In [32]:
def load_and_prepare_model(model, freeze_layers=None):
    """
    Load and prepare a transformer model with specified layer freezing
    
    Args:
        model_name (str): Name of the pretrained model
        freeze_layers (int): Number of layers to freeze from the bottom
    """    
    # Get transformer component
    transformer = model._first_module().auto_model
    
    # Count layers
    total_layers = len([name for name, _ in transformer.named_parameters() 
                       if "encoder.layer" in name and ".attention" in name]) // 4
    
    print(f"Total transformer layers: {total_layers}")
    
    if freeze_layers is not None:
        # Validate freeze_layers
        if freeze_layers >= total_layers:
            raise ValueError(f"freeze_layers ({freeze_layers}) must be less than total layers ({total_layers})")
        
        # Freeze specified number of layers
        for name, param in transformer.named_parameters():
            # Check if parameter is in a layer that should be frozen
            layer_match = any(f"encoder.layer.{i}." in name 
                            for i in range(freeze_layers))
            
            if layer_match:
                param.requires_grad = False
                print(f"Frozen: {name}")
            else:
                param.requires_grad = True
                print(f"Trainable: {name}")
    
    # Verify freezing
    trainable_params = sum(p.numel() for p in transformer.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in transformer.parameters())
    print(f"\nTrainable parameters: {trainable_params:,}")
    print(f"Total parameters: {total_params:,}")
    print(f"Frozen parameters: {total_params - trainable_params:,}")

With the previous function, we freeze the first 10 layers of the model to only affect the last two layers during training.


In [33]:
load_and_prepare_model(model, freeze_layers=10)

Total transformer layers: 12
Trainable: embeddings.word_embeddings.weight
Trainable: embeddings.LayerNorm.weight
Trainable: embeddings.LayerNorm.bias
Frozen: encoder.layer.0.attention.qkv_proj.weight
Frozen: encoder.layer.0.attention.qkv_proj.bias
Frozen: encoder.layer.0.attention.o_proj.weight
Frozen: encoder.layer.0.attention.o_proj.bias
Frozen: encoder.layer.0.mlp.up_gate_proj.weight
Frozen: encoder.layer.0.mlp.down_proj.weight
Frozen: encoder.layer.0.mlp.down_proj.bias
Frozen: encoder.layer.0.attn_ln.weight
Frozen: encoder.layer.0.attn_ln.bias
Frozen: encoder.layer.0.mlp_ln.weight
Frozen: encoder.layer.0.mlp_ln.bias
Frozen: encoder.layer.1.attention.qkv_proj.weight
Frozen: encoder.layer.1.attention.qkv_proj.bias
Frozen: encoder.layer.1.attention.o_proj.weight
Frozen: encoder.layer.1.attention.o_proj.bias
Frozen: encoder.layer.1.mlp.up_gate_proj.weight
Frozen: encoder.layer.1.mlp.down_proj.weight
Frozen: encoder.layer.1.mlp.down_proj.bias
Frozen: encoder.layer.1.attn_ln.weight
Froze

We use the `TripletEvaluator` class from `SentenceTransformer` to evaluate the model with the validation dataset and the `TripletLoss` class to compute the triplet loss.


In [34]:
loss = losses.TripletLoss(model=model, distance_metric=losses.TripletDistanceMetric.COSINE)
evaluator = TripletEvaluator(
    name = 'evaluator',
    anchors=evaluate['anchor'],
    positives=evaluate['positive'],
    negatives=evaluate['negative'],
    batch_size=16,
)

We use the `SentenceTransformerTrainingArguments` class where we specify the training arguments:


### Basic Hyperparameters:

* `EPOCHS = X`: The model will see the complete training data X times.
* `WARMUP_STEPS`: Calculated as 10% of the total training steps (dataset size * epochs). During these initial steps the learning rate increases gradually.
* `BATCH_SIZE = 16`: Number of samples processed in parallel.

### Main Training Arguments:

* `output_dir and logging_dir`: Directories to save the model and logs respectively.
* `learning_rate=1e-5`: Relatively small learning rate for stable fine-tuning.
* `weight_decay=0.01`: Regularization to prevent overfitting.
* `gradient_accumulation_steps=2`: Accumulates gradients from 2 batches before updating weights, simulating larger batches.

### Evaluation and Checkpoints:

* `eval_strategy and save_strategy="steps"`: Evaluates and saves the model every Y steps.
* `load_best_model_at_end=True`: At the end, loads the best model according to the evaluation metric.
* `metric_for_best_model="eval_evaluator_cosine_accuracy"`: Uses cosine similarity accuracy as the metric to select the best model.
* `resume_from_checkpoint=True`: Allows continuing training from a checkpoint if interrupted.


In [35]:
EPOCHS = 5
WARMUP_STEPS = int(len(train) * 0.1 * EPOCHS)
BATCH_SIZE = 16

In [36]:
training_args = SentenceTransformerTrainingArguments(
    output_dir="./output",
    num_train_epochs=EPOCHS,
    warmup_steps= WARMUP_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,
    logging_dir="./logs",
    learning_rate=1e-5,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    resume_from_checkpoint= True,
    metric_for_best_model="eval_evaluator_cosine_accuracy",
    greater_is_better=True,
    do_train=True,
    do_eval=True,
)

Finally, as a best practice, we add EarlyStopping to halt training if the evaluation metric does not improve after a certain number of steps. However, we did not use it in this case, as the training is relatively short and we did not want it to stop too early.


In [37]:
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=3,
    early_stopping_threshold=0.01
)

We instantiate the trainer to train the model. In our case, the training was done in a notebook on [Kaggle](https://www.kaggle.com/code/alejandroparbas/transformer), as they offered free compute hours which we took advantage of.


In [ ]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train,
    evaluator=evaluator,
    loss=loss,
    callbacks=[early_stopping]
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("nlp_embeddings")

The models were trained using the Kaggle platform in this [notebook](https://www.kaggle.com/code/alejandroparbas/transformer), and 3 different versions were trained: one for 3 epochs (run 15), another for 10 epochs (run 16), and another for 20 epochs (run 9), to see which model to select.

The best model with 20 epochs is available at [HuggingFace](https://huggingface.co/Alejandro-Pardo/voight-kampff-pan2024-gte-en-v1.5)


These are the training metrics obtained for each model:

- `3 epochs`:

| Epoch      | Step    | Training Loss | Validation Loss | evaluator_cosine_accuracy |
|:----------:|:-------:|:-------------:|:---------------:|:-------------------------:|
| 0.5556     | 100     | -             | 5.0064          | 0.4661                    |
| 1.1111     | 200     | -             | 5.0058          | 0.4679                    |
| 1.6667     | 300     | -             | 5.0049          | 0.4711                    |
| 2.2222     | 400     | -             | 5.0036          | 0.4764                    |
| 2.7778     | 500     | 5.0044        | 5.0019          | 0.4843                    |
| **3.3333** | **600** | **-**         | **4.9998**      | **0.4921**                |

- `10 epochs`:

| Epoch    | Step     | Training Loss | Validation Loss | evaluator_cosine_accuracy |
|:--------:|:--------:|:-------------:|:---------------:|:-------------------------:|
| 0.5556   | 100      | -             | 5.0064          | 0.4661                    |
| 1.1111   | 200      | -             | 5.0058          | 0.4679                    |
| 1.6667   | 300      | -             | 5.0049          | 0.4711                    |
| 2.2222   | 400      | -             | 5.0036          | 0.4764                    |
| 2.7778   | 500      | 5.0044        | 5.0019          | 0.4843                    |
| 3.3333   | 600      | -             | 4.9998          | 0.4921                    |
| 3.8889   | 700      | -             | 4.9972          | 0.5023                    |
| 4.4444   | 800      | -             | 4.9939          | 0.514                     |
| 5.0      | 900      | -             | 4.9898          | 0.5303                    |
| 5.5556   | 1000     | 4.995         | 4.9845          | 0.5485                    |
| 6.1111   | 1100     | -             | 4.9772          | 0.5759                    |
| 6.6667   | 1200     | -             | 4.9649          | 0.6150                    |
| 7.2222   | 1300     | -             | 4.9371          | 0.6655                    |
| 7.7778   | 1400     | -             | 4.8736          | 0.7313                    |
| 8.3333   | 1500     | 4.9411        | 4.7912          | 0.7681                    |
| 8.8889   | 1600     | -             | 4.7078          | 0.7872                    |
| 9.4444   | 1700     | -             | 4.6172          | 0.8016                    |
| **10.0** | **1800** | **-**         | **4.5268**      | **0.8052**                |

- `20 epochs`:

| Epoch       | Step     | Training Loss | Validation Loss | evaluator_cosine_accuracy |
|:-----------:|:--------:|:-------------:|:---------------:|:-------------------------:|
| 2.7778      | 500      | 5.0051        | 5.0043          | 0.4734                    |
| 5.5556      | 1000     | 5.0008        | 4.9969          | 0.5029                    |
| 8.3333      | 1500     | 4.9902        | 4.9804          | 0.5650                    |
| 11.1111     | 2000     | 4.953         | 4.8645          | 0.7361                    |
| 13.8889     | 2500     | 4.7692        | 4.5741          | 0.8066                    |
| 16.6667     | 3000     | 4.4897        | 4.2532          | 0.8212                    |
| **19.4444** | **3500** | **4.2394**    | **4.0644**      | **0.8324**                |



Now that we have the trained models, let's load them and make predictions. We will see how, although the other models have been trained for longer, the 3-epoch model practically obtains the same results as the 10-epoch model, with a greater loss reduction for the 20-epoch model.

We believe this is due to the triplet evaluator that uses cosine accuracy as a metric, which is indicating that the anchor and positive sentences are getting too close.

Therefore, our hypothesis is that the 3-epoch model will perform better, as the other models may have been overfitting, and despite the loss decreasing, they are not learning anything new, since the cosine accuracy grows too quickly. It would be interesting to try with a lower learning rate to see if the model learns more slowly and does not increase cosine accuracy so rapidly.

Nevertheless, we will fit a classifier for each of the models we trained to see which one yields the best results.


## SVM Training for the Embedding Generators


We will now use the pre-trained embedding models to train a text classifier. For this, we use a Support Vector Machine with a linear kernel, since the embeddings already exhibit high linearity. Additionally, we use a calibrator to obtain prediction probabilities, calibrating the model with the test dataset. This is used to return prediction confidence as in the original competition.

Now we will see, with the results of our final classifiers, how the f1-score of the 3-epoch model holds up against the others, despite having been trained for fewer intervals.


In [2]:
from sklearn.metrics import classification_report
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC
from sentence_transformers import SentenceTransformer
import numpy as np
import pickle as pkl

Since we are not using cross-validation, we will use both the training and validation sets to train the models, in order to obtain a more robust model. Finally, to calibrate the model, we will use the test set.


In [7]:
train_tuples =  generators["train"].generate_singles()
validate_tuples = generators["validate"].generate_singles()
combined_tuples = train_tuples + validate_tuples
test_tuples = generators["test"].generate_singles()

# Extraer textos y etiquetas
train_texts, train_labels = zip(*combined_tuples)
test_texts, test_labels = zip(*test_tuples)

### 20-Epoch Embedding Model


For each model, we first load the embedding model and use it to generate embeddings for the training and validation text sets. Then, we train a Support Vector Machine classifier with a linear kernel using the generated embeddings and corresponding labels. Finally, we calibrate the model with the test set.

Additionally, we obtain the model metrics such as f1-score, precision, and recall on the test set. It should be noted that **these results are based on sentences of ~64 tokens**, since we preprocessed the texts to be divided into chunks of ~32 tokens and took chunks with size 2 token_size **(2 * 32 tkns = ~64 tokens)**.

Therefore, since we are not directly classifying complete texts, the results obtained do not reflect the final model performance. Later we will see how, in the classification of complete texts, we obtain a weighted average of the chunk predictions, where we expect the precision on complete texts to improve.


In [42]:
model = SentenceTransformer('nlp_embeddings_20', device='cuda', trust_remote_code=True)
train_embeddings = model.encode(train_texts, convert_to_tensor=True, show_progress_bar=True, batch_size=32)
test_embeddings = model.encode(test_texts, convert_to_tensor=True, show_progress_bar=True, batch_size=32)
train_embeddings = np.array(train_embeddings.cpu())
test_embeddings = np.array(test_embeddings.cpu())

Batches:   0%|          | 0/719 [00:00<?, ?it/s]

Batches:   0%|          | 0/360 [00:00<?, ?it/s]

In [ ]:
svm = LinearSVC(random_state=42, max_iter=10000)
svm.fit(train_embeddings, train_labels)

LinearSVC(random_state=42)

In [44]:
calibrated_linear_svc = CalibratedClassifierCV(estimator=svm, n_jobs=-1, cv='prefit')
calibrated_linear_svc.fit(test_embeddings, test_labels)
test_predictions = calibrated_linear_svc.predict(test_embeddings)

In [46]:
report = classification_report(test_labels, test_predictions)
print("Classification Report:")
print(report)

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.84      0.83      5753
           1       0.84      0.81      0.82      5747

    accuracy                           0.82     11500
   macro avg       0.83      0.82      0.82     11500
weighted avg       0.83      0.82      0.82     11500



In [ ]:
with open("nlp_classifier_20.pkl", "wb") as f_out:
    pkl.dump(calibrated_linear_svc, f_out)

### 10-Epoch Embedding Model


In [8]:
model = SentenceTransformer('nlp_embeddings_10', device='cuda', trust_remote_code=True)
train_embeddings = model.encode(train_texts, convert_to_tensor=True, show_progress_bar=True, batch_size=32)
test_embeddings = model.encode(test_texts, convert_to_tensor=True, show_progress_bar=True, batch_size=32)
train_embeddings = np.array(train_embeddings.cpu())
test_embeddings = np.array(test_embeddings.cpu())

Batches:   0%|          | 0/719 [00:00<?, ?it/s]

Batches:   0%|          | 0/360 [00:00<?, ?it/s]

In [ ]:
svm = LinearSVC(random_state=42, max_iter=10000)
svm.fit(train_embeddings, train_labels)

LinearSVC(random_state=42)

In [10]:
calibrated_linear_svc = CalibratedClassifierCV(estimator=svm, n_jobs=-1, cv='prefit')
calibrated_linear_svc.fit(test_embeddings, test_labels)
test_predictions = calibrated_linear_svc.predict(test_embeddings)

In [11]:
report = classification_report(test_labels, test_predictions)
print("Classification Report:")
print(report)

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.84      0.82      5751
           1       0.83      0.80      0.82      5749

    accuracy                           0.82     11500
   macro avg       0.82      0.82      0.82     11500
weighted avg       0.82      0.82      0.82     11500



In [ ]:
# Save the model
with open("nlp_classifier_10.pkl", "wb") as f_out:
    pkl.dump(calibrated_linear_svc, f_out)

### 3-Epoch Embedding Model


In [ ]:
model = SentenceTransformer('nlp_embeddings_3', device='cuda', trust_remote_code=True)

train_embeddings = model.encode(train_texts, convert_to_tensor=True, show_progress_bar=True, batch_size=32)
test_embeddings = model.encode(test_texts, convert_to_tensor=True, show_progress_bar=True, batch_size=32)
train_embeddings = np.array(train_embeddings.cpu())
test_embeddings = np.array(test_embeddings.cpu())

Batches:   0%|          | 0/719 [00:00<?, ?it/s]

Batches:   0%|          | 0/360 [00:00<?, ?it/s]

In [ ]:
svm = LinearSVC(random_state=42, max_iter=10000)
svm.fit(train_embeddings, train_labels)

LinearSVC(random_state=42)

In [53]:
calibrated_linear_svc = CalibratedClassifierCV(estimator=svm, n_jobs=-1, cv='prefit')
calibrated_linear_svc.fit(test_embeddings, test_labels)

CalibratedClassifierCV(cv='prefit', estimator=LinearSVC(random_state=42),
                       n_jobs=-1)

In [54]:
test_predictions = calibrated_linear_svc.predict(test_embeddings)
report = classification_report(test_labels, test_predictions)
print("Classification Report:")
print(report)

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.83      0.81      5753
           1       0.82      0.77      0.79      5747

    accuracy                           0.80     11500
   macro avg       0.80      0.80      0.80     11500
weighted avg       0.80      0.80      0.80     11500



In [ ]:
# Save the model
with open("nlp_classifier_3.pkl", "wb") as f_out:
    pkl.dump(calibrated_linear_svc, f_out)

With the results obtained, we achieve for all three embedding models an f1-score of around 80% on text chunks for classifying them as AI or human. In principle, by taking a random chunk from the text, we would be able to correctly classify 80% of the texts presented to the model, surpassing the competition baseline (without considering the modifications introduced to the testing dataset).

Interestingly, the scores of the three classifiers we trained are very similar, contrary to what we previously thought with the embedding models. We will see in the evaluation section whether these classifiers differentiate better with the complete classifier implementation, which we will see in the Evaluation section with complete texts.


## Evaluation of the Proposed Model


### Evaluation Code


First, we use the evaluation functions indicated for the competition. We use `sklearn.metrics` functions for some metrics (such as `F1 score`, `ROC-AUC`, and `Brier Score`) and use the competition's implementation for `c@1 score` and `F0.5u score`. The original competition code can be found in their [GitHub Repository](https://github.com/pan-webis-de/pan-code/blob/master/clef24/generative-authorship-verification/pan24_llm_evaluator/pan24_llm_evaluator/evaluator.py#L161).


In [4]:
import numpy as np
import pickle as pkl
import re
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss

def auc(true_y, pred_y):
    """
    Calculates the ROC-AUC score (Area Under the Curve).

    ROC-AUC is a well-known scalar evaluation score for binary classifiers.

    Parameters
    ----------
    true_y : array [n_problems]
        The predictions of a verification system.
        Assumes `0 >= prediction <=1`.
    pred_y : array [n_problems]
        The gold annotations provided for each problem as binary labels.

    Returns
    ----------
    The Area Under the ROC Curve.
    """
    return roc_auc_score(true_y, pred_y)

def c_at_1(true_y, pred_y):
    """
    Calculates the c@1 score.

    This method rewards predictions which leave some problems
    unanswered (score = 0.5). See:
        A. Peñas and A. Rodrigo. A Simple Measure to Assess Nonresponse.
        In Proc. of the 49th Annual Meeting of the Association for
        Computational Linguistics, Vol. 1, pages 1415-1424, 2011.

    Parameters
    ----------
    true_y : array [n_problems]
        The predictions of a verification system.
        Assumes `0 >= prediction <=1`, unanswered problems = 0.5.
    pred_y : array [n_problems]
        The gold annotations provided for each problem as binary labels.

    Returns
    ----------
    The c@1 measure.
    """

    nu = pred_y == 0.5
    nc = np.sum((true_y == (pred_y > 0.5))[~nu])
    nu = np.sum(nu)
    return (1 / len(true_y)) * (nc + (nu * nc / len(true_y)))

def f1(true_y, pred_y):
    """
    Calculates the F1 score.

    Assesses verification performance, assuming that every score > 0.5
    represents a same-author pair decision.

    Parameters
    ----------
    true_y : array [n_problems]
        The predictions of a verification system.
        Assumes `0 >= prediction <=1`.
    pred_y : array [n_problems]
        The gold annotations provided for each problem as binary labels.

    Returns
    ----------
    The F1 score.

    References
    ----------
        E. Stamatatos, et al. Overview of the Author Identification
        Task at PAN 2014. CLEF (Working Notes) 2014: 877-897.
    """

    return f1_score(true_y, pred_y > 0.5)

def f05u(true_y, pred_y):
    """
    Calculates the F0.5u score.

    F0.5u is a modified F0.5 measure which treats unanswered problems as false negatives.
    as false negatives. See:
        J. Bevendorff et al. Generalizing unmasking for short texts.
        In Proc. of the 14th Conference of the North American Chapter of
        the Association for Computational Linguistics (NAACL 2019). Pages 654–659.

    Parameters
    ----------
    true_y : array [n_problems]
        The predictions of a verification system.
        Assumes `0 >= prediction <=1`, unanswered problems = 0.5
    pred_y : array [n_problems]
        The gold annotations provided for each problem as binary labels.

    Returns
    ----------
    The F0.5u score.
    """

    n_tp = np.sum(true_y * (pred_y > 0.5))
    n_fn = np.sum(true_y * (pred_y < 0.5))
    n_fp = np.sum((1.0 - true_y) * (pred_y > 0.5))
    n_u = np.sum(pred_y == 0.5)

    return (1.25 * n_tp) / (1.25 * n_tp + 0.25 * (n_fn + n_u) + n_fp)

def brier_score(true_y, pred_y):
    """
    Calculates the complement of the Brier score loss (which is bounded
    to [0-1]), so that higher scores indicate better performance.
    We use the Brier implementation in scikit-learn [Pedregosa et al. 2011].

    Parameters
    ----------
    true_y : array [n_problems]
        The predictions of a verification system.
        Assumes `0 >= prediction <=1`.
    pred_y : array [n_problems]
        The gold annotations provided for each problem as binary labels.

    Returns
    ----------
    Complement of the Brier score.
    """
    try:
        return 1 - brier_score_loss(true_y, pred_y)
    except ValueError:
        return 0.0

def evaluate_all(true_y, pred_y):
    """
    Calculate all evaluation scores and return results as a dict.
    All scores are rounded to three digits.

    :param true_y : truth as numpy array [n_problems]
    :param pred_y : predictions as numpy array [n_problems]
    """
    results = {
        'roc-auc': auc(true_y, pred_y),
        'brier': brier_score(true_y, pred_y),
        'c@1': c_at_1(true_y, pred_y),
        'f1': f1(true_y, pred_y),
        'f05u': f05u(true_y, pred_y)
    }
    results['mean'] = np.mean(list(results.values()))
    for k, v in results.items():
        results[k] = round(v, 3)
    return results

We also implement the `TextClassification` class, where you first specify the text preprocessing details (`train_tk_size` == `token_size` == 32, `train_tolerance` == `tolerance` = 2, and `train_chunk_size` == `token_size` = 2) and the names of the saved models for evaluation. `device` specifies whether to use `cuda` or the default device.


In [5]:
class TextClassification:
    def __init__(self,
                 train_tk_size: int=32,
                 train_tolerance: int=2,
                 train_chunk_size: int=2,
                 device: str=None,
                 embeddings_name: str='nlp_embeddings_3',
                 classifier_name: str='nlp_classifier_3.pkl'):
        self.train_tk_size = train_tk_size
        self.train_tolerance = train_tolerance
        self.train_chunk_size = train_chunk_size
        self.embedding_model = SentenceTransformer(embeddings_name, device=device, trust_remote_code=True)
        self.tokenizer = self.embedding_model.tokenizer
        with open(classifier_name, "rb") as f_in:
            self.classifier_model = pkl.load(f_in)

    def chunk_text(self, text: str):
        """
        Processes the passed text and obtains the token chunks to save with the document.
        """
        variance = int(self.train_tk_size / self.train_tolerance)
        tk_variance = (self.train_tk_size - variance, self.train_tk_size + variance)
        text_positions = [0]
        last_position = 0
        prev_token = 0
        for phrase in re.split(r"[.!?\n\r]", text):
            if len(phrase) == 0:
                last_position += 1
                continue
            curr_position = last_position + len(phrase) + 1
            curr_tokens = len(self.tokenizer.tokenize(text[text_positions[-1]:curr_position]))
            if prev_token > tk_variance[0] and curr_tokens > tk_variance[1]:
                text_positions.append(last_position)
                prev_token = curr_tokens - prev_token
            else:
                prev_token = curr_tokens
            last_position += len(phrase) + 1
        if prev_token != 0:
            text_positions.append(last_position)
        if len(text_positions) <= self.train_chunk_size:
            return [text]
        return [text[text_positions[k - self.train_chunk_size]:text_positions[k]]
            for k in range(self.train_chunk_size, len(text_positions))
        ]

    def classify_texts(self, text_comparisons):
        classified_texts = []
        for idx, text_comp in enumerate(text_comparisons):
            if "id" not in text_comp:
                text_comp["id"] = f"comparison-{idx}"
            if idx % 100 == 0:
                print(f"Evaluation {idx:5d} | Progress {100 * idx / len(text_comparisons):.2f}%")
            embedding_text1 = self.embedding_model.encode(self.chunk_text(text_comp["text1"]), convert_to_tensor=True).cpu()
            embedding_text2 = self.embedding_model.encode(self.chunk_text(text_comp["text2"]), convert_to_tensor=True).cpu()
            # As we now one of the two is AI, we know that if t1 is human, t2 must be AI
            is_human1 = np.mean(self.classifier_model.predict_proba(embedding_text1)[:, 0]) # If is_human, the p > 0.5
            is_llm2 = np.mean(self.classifier_model.predict_proba(embedding_text2)[:, 1])
            # As if the t1 is human, is_human < 0.5, and if t2 is human, is_human > 0.5
            classified_texts.append({"id": text_comp["id"], "is_human": np.clip(((1 - is_human1) + (1 - is_llm2)) / 2, 0.0, 1.0)})
            del embedding_text1
            del embedding_text2
        return classified_texts

Looking at the `classify_texts` function, we use the format in which texts are passed to the function, which is a list with the following structure:
```json
[
    {"id": "iixcWBmKWQqLAwVXxXGBGg", "text1": "...", "text2": "..."},
    {"id": "y12zUebGVHSN9yiL8oRZ8Q", "text1": "...", "text2": "..."},
    ...
]
```
The expected output is a list with the comparison `id` and an `is_human` value, where `is_human < 0.5` indicates higher confidence that text 1 is human, `is_human = 0.5` means it cannot decide, and `is_human > 0.5` indicates higher confidence that text 2 is human.
```json
[
    {"id": "iixcWBmKWQqLAwVXxXGBGg", "is_human": 1.0},
    {"id": "y12zUebGVHSN9yiL8oRZ8Q", "is_human": 0.3},
    ...
]
```
In our case, if text pairs do not have an `id`, we generate a default one, `comparison-%d`. To classify each text pair, we perform the following process:

1. We chunk `text1` and `text2` using the embedding model's `tokenizer`, with sizes `train_tk_size`, `train_tolerance`, and `train_chunk_size`. First we obtain the list of text positions, `text_positions`, based on `train_tk_size` and `train_tolerance` following the same process as the `DataLoader` class. We obtain `len(text_positions) - train_chunk_size` chunks of text (`p` = `text_positions`):
$$text_{chunks} = [text[p[0]: p[chunk_{size}]], text[p[1]: p[chunk_{size} + 1]], ..., text[p[len(p) - chunk_{size}]: p[len(p)]]]$$

2. With the chunks of each text, we process them into their embedding representations.
3. With their embeddings, we classify each of them with probabilities and for the first text we obtain the average probability of being human (value 0 of the classification), and for the second the average probability of being an LLM.
4. We modify the formula used in the `baselineavengers` paper to adapt it to our needs. The results are returned in the same format as requested in the competition, so we need to modify the probabilities we obtain.
  
  Since the probability that text 1 is human, if high, will be greater than 0.5, we use 1 - P(human|t1), meaning t1 has a high possibility of being human the higher P(human|t1) is. For P(LLM|t2), we also invert it, since the higher the probability of P(LLM|t2), the higher the probability that t1 is human, so we consider 1 - P(LLM|t2). Thus the `is_human` value is calculated with the following formula:
$$is\_human = \frac{(1 - P(human|t_1)) + (1 - P(LLM|t_2))}{2}$$




### Evaluation with Same Noise as Training


First, we obtain the generator with the parameters used for training with the documents selected for testing. We obtain random pairings of test texts (human-AI, AI-human), with the score `is_human = 0.0` if t1 is human and `is_human = 1.0` if text2 is human.


In [ ]:
import os
import json

os.environ["TOKENIZERS_PARALLELISM"] = "false"
loader = DatasetLoader(
    storage_location="documents",
    storage_information={
        "tokenizer_name": "Alibaba-NLP/gte-base-en-v1.5",
        "dataset_types": {"train": 0.75, "validate": 0.15, "test": 0.1}
    }
)
generators = loader.create_generators(
    n_test_cases=1000,
    token_size=2,
    rep_noisy_cases=0.15,
    change_prob=0.2,
    change_freq=0.2
)
text_comparisons = generators["test"].generate_random_pairings()
y_true = np.array([text_comp["is_human"] for text_comp in text_comparisons])

print("---------------------------------------")
print(json.dumps(text_comparisons[0], indent=4))
print("---------------------------------------")
print(json.dumps(text_comparisons[500], indent=4))
print("---------------------------------------")
print(json.dumps(text_comparisons[-1], indent=4))

---------------------------------------
{
    "text1": "Trump's QAnon Supporters Thought JFK Jr., Famously Dead, Was Going to Show Up in Texas Today\n\nThere are a lot of reasons Donald Trump lost the last election, chief among them being his disastrous handling of the COVID-19 pandemic, which resulted in more than 232,000 deaths by the time voters went to the polls one year ago. But we're pretty sure another contributor was the fact that he had become even more of a batshit crazy lunatic than usual, to the point that he was going on TV and suggesting his political enemies were running a satanic pedophile cult.\n\nIf you blocked that out of your mind to make room for all of the other insane events that went down over the last year, here's a quick refresher. On October 15, 2020, after backing out of the second presidential debate, Trump sat down with NBC's Savannah Guthrie for a live town hall. At one point, Guthrie said, \"Let me ask you about QAnon,\" referring to the far-right, incom

We perform the evaluation for all models and save them in a table for comparison.


In [11]:
classifier = TextClassification(
    device=None,
    embeddings_name='nlp_embeddings_3',
    classifier_name='nlp_classifier_3.pkl'
)

classified_texts3 = classifier.classify_texts(text_comparisons)
y_pred = np.array([text_comp["is_human"] for text_comp in classified_texts3])
eval3 = evaluate_all(y_true, y_pred)

print(f"Evaluation:\n{json.dumps(eval3, indent=4)}")

Evaluation     0 | Progress 0.00%
Evaluation   100 | Progress 8.70%
Evaluation   200 | Progress 17.39%
Evaluation   300 | Progress 26.09%
Evaluation   400 | Progress 34.78%
Evaluation   500 | Progress 43.48%
Evaluation   600 | Progress 52.17%
Evaluation   700 | Progress 60.87%
Evaluation   800 | Progress 69.57%
Evaluation   900 | Progress 78.26%
Evaluation  1000 | Progress 86.96%
Evaluation  1100 | Progress 95.65%
Evaluation:
{
    "roc-auc": 0.985,
    "brier": 0.905,
    "c@1": 0.93,
    "f1": 0.929,
    "f05u": 0.937,
    "mean": 0.938
}


In [12]:
classifier = TextClassification(
    device=None,
    embeddings_name='nlp_embeddings_10',
    classifier_name='nlp_classifier_10.pkl'
)

classified_texts10 = classifier.classify_texts(text_comparisons)
y_pred = np.array([text_comp["is_human"] for text_comp in classified_texts10])
eval10 = evaluate_all(y_true, y_pred)

print(f"Evaluation:\n{json.dumps(eval10, indent=4)}")

Evaluation     0 | Progress 0.00%
Evaluation   100 | Progress 8.70%
Evaluation   200 | Progress 17.39%
Evaluation   300 | Progress 26.09%
Evaluation   400 | Progress 34.78%
Evaluation   500 | Progress 43.48%
Evaluation   600 | Progress 52.17%
Evaluation   700 | Progress 60.87%
Evaluation   800 | Progress 69.57%
Evaluation   900 | Progress 78.26%
Evaluation  1000 | Progress 86.96%
Evaluation  1100 | Progress 95.65%
Evaluation:
{
    "roc-auc": 0.991,
    "brier": 0.915,
    "c@1": 0.944,
    "f1": 0.944,
    "f05u": 0.946,
    "mean": 0.948
}


In [13]:
classifier = TextClassification(
    device=None,
    embeddings_name='nlp_embeddings_20',
    classifier_name='nlp_classifier_20.pkl'
)

classified_texts20 = classifier.classify_texts(text_comparisons)
y_pred = np.array([text_comp["is_human"] for text_comp in classified_texts20])
eval20 = evaluate_all(y_true, y_pred)

print(f"Evaluation:\n{json.dumps(eval20, indent=4)}")

Evaluation     0 | Progress 0.00%
Evaluation   100 | Progress 8.70%
Evaluation   200 | Progress 17.39%
Evaluation   300 | Progress 26.09%
Evaluation   400 | Progress 34.78%
Evaluation   500 | Progress 43.48%
Evaluation   600 | Progress 52.17%
Evaluation   700 | Progress 60.87%
Evaluation   800 | Progress 69.57%
Evaluation   900 | Progress 78.26%
Evaluation  1000 | Progress 86.96%
Evaluation  1100 | Progress 95.65%
Evaluation:
{
    "roc-auc": 0.993,
    "brier": 0.924,
    "c@1": 0.951,
    "f1": 0.951,
    "f05u": 0.953,
    "mean": 0.955
}


In [14]:
import pandas as pd

evaluation = pd.DataFrame.from_records([eval3, eval10, eval20], index=["3-epoch-noisy-train", "5-epoch-noisy-train", "20-epoch-noisy-train"])
evaluation

,roc-auc,brier,c@1,f1,f05u,mean
3-epoch-noisy-train,0.985,0.905,0.930,0.929,0.937,0.938
5-epoch-noisy-train,0.991,0.915,0.944,0.944,0.946,0.948
20-epoch-noisy-train,0.993,0.924,0.951,0.951,0.953,0.955


Evaluation Results:
```json
epoch-3 noisy
{
    "roc-auc": 0.985,
    "brier": 0.905,
    "c@1": 0.93,
    "f1": 0.929,
    "f05u": 0.937,
    "mean": 0.938
}

epoch-10 noisy
{
    "roc-auc": 0.991,
    "brier": 0.915,
    "c@1": 0.944,
    "f1": 0.944,
    "f05u": 0.946,
    "mean": 0.948
}

epoch-20 noisy
{
    "roc-auc": 0.993,
    "brier": 0.924,
    "c@1": 0.951,
    "f1": 0.951,
    "f05u": 0.953,
    "mean": 0.955
}
```


Based on the scores obtained, the 20-epoch-noisy-train model manages to improve upon the scores obtained in the competition, although we cannot compare them one-to-one since we do not have the test dataset used to compare models, and we have not tested with German texts in training or testing.

Even so, we achieve quite good results with the 20-epoch embedding model, achieving improvements over the 3-epoch model, although the difference between 10 and 20 epochs is not as significant, it does show improvement. This allows us to see that our assumption that we were overfitting the model was wrong, as there is a measurable improvement between models with different epochs.

We suspect the reason the three classifiers had similar scores is that each classifier only considered the embeddings for classification, so the quality of the embeddings was not taken into account. Therefore, since the embedding quality of models with more epochs was higher, the text classification was better.

One limitation of our results is that since we are not using an embedding model specifically trained to be multilingual, it could face the same limitations that the competition models faced, where their scores were penalized due to not being able to adapt to multilingual inputs.

We also note that our model will perform better when the texts it faces are large, since the more chunks we have to classify, the higher our confidence in whether the text is AI or not. But if we have few chunks (1 or 2), as we saw previously, the sentence classifier had an accuracy of 80%, so if both texts are small, the results obtained have a high probability of being erroneous.

Additionally, we believe our model is more robust against noise such as the leetspeak introduced in the competition's testing cases, since we introduced this variable during embedding training.

In the future, it would be interesting to try training a multilingual embedding model with a more diverse dataset in several languages and introducing more noise during training to observe how it would improve our model's robustness.

We include here the competition results for comparison with our own.

| #   | Team                   | System                 | ROC-AUC | Brier | C@1   | F1    | F0.5u | Mean  |
|-----|------------------------|------------------------|---------|-------|-------|-------|-------|-------|
| 1   | marsan                | staff-trunk           | 0.961   | 0.928 | 0.912 | 0.884 | 0.932 | 0.924 |
| 2   | you-shun-you-de       | charitable-mole_v3    | 0.931   | 0.926 | 0.928 | 0.905 | 0.913 | 0.921 |
| 3   | baselineavengers      | svm                   | 0.925   | 0.869 | 0.882 | 0.875 | 0.869 | 0.886 |
| 4   | g-fosunlpteam         | gritty-producer       | 0.889   | 0.875 | 0.887 | 0.884 | 0.884 | 0.884 |
| 5   | lam                   | blistering-moss       | 0.851   | 0.850 | 0.850 | 0.852 | 0.849 | 0.851 |
| 6   | drocks                | muffled-stock         | 0.866   | 0.863 | 0.834 | 0.825 | 0.820 | 0.843 |
| 7   | aida                  | corporate-burn        | 0.831   | 0.825 | 0.795 | 0.788 | 0.782 | 0.806 |
| 8   | cnlp-nits-pp          | direct-velocity       | 0.844   | 0.793 | 0.805 | 0.789 | 0.792 | 0.806 |
| 9   | fosu-stu              | proud-stick           | 0.833   | 0.867 | 0.799 | 0.748 | 0.767 | 0.804 |
| 10  | ap-team               | marinated-pantone     | 0.853   | 0.862 | 0.795 | 0.718 | 0.742 | 0.796 |
| 11  | heartsteel            | canary-paint          | 0.777   | 0.777 | 0.777 | 0.780 | 0.777 | 0.778 |
| 12  | verification-team     | merciless-lease       | 0.799   | 0.788 | 0.740 | 0.740 | 0.741 | 0.763 |
| -   | Baseline Binoculars (Falcon-7B) | -       | 0.751   | 0.780 | 0.734 | 0.720 | 0.720 | 0.741 |



### Evaluation without Noise


We also perform evaluations without noise to verify that it is not significantly affecting the results obtained.


In [ ]:
import os
import json

os.environ["TOKENIZERS_PARALLELISM"] = "false"
loader = DatasetLoader(
    storage_location="documents",
    storage_information={
        "tokenizer_name": "Alibaba-NLP/gte-base-en-v1.5",
        "dataset_types": {"train": 0.75, "validate": 0.15, "test": 0.1}
    }
)
generators = loader.create_generators(
    n_test_cases=1150,
    token_size=2,
    rep_noisy_cases=0.0,
    change_prob=0.0,
    change_freq=0.0
)
text_comparisons = generators["test"].generate_random_pairings()
y_true = np.array([text_comp["is_human"] for text_comp in text_comparisons])

In [7]:
classifier = TextClassification(
    device=None,
    embeddings_name='nlp_embeddings_3',
    classifier_name='nlp_classifier_3.pkl'
)

classified_texts3 = classifier.classify_texts(text_comparisons)
y_pred = np.array([text_comp["is_human"] for text_comp in classified_texts3])
eval3 = evaluate_all(y_true, y_pred)

print(f"Evaluation:\n{json.dumps(eval3, indent=4)}")

Evaluation     0 | Progress 0.00%
Evaluation   100 | Progress 8.70%
Evaluation   200 | Progress 17.39%
Evaluation   300 | Progress 26.09%
Evaluation   400 | Progress 34.78%
Evaluation   500 | Progress 43.48%
Evaluation   600 | Progress 52.17%
Evaluation   700 | Progress 60.87%
Evaluation   800 | Progress 69.57%
Evaluation   900 | Progress 78.26%
Evaluation  1000 | Progress 86.96%
Evaluation  1100 | Progress 95.65%
Evaluation:
{
    "roc-auc": 0.986,
    "brier": 0.906,
    "c@1": 0.933,
    "f1": 0.933,
    "f05u": 0.937,
    "mean": 0.939
}


In [8]:
classifier = TextClassification(
    device=None,
    embeddings_name='nlp_embeddings_10',
    classifier_name='nlp_classifier_10.pkl'
)

classified_texts10 = classifier.classify_texts(text_comparisons)
y_pred = np.array([text_comp["is_human"] for text_comp in classified_texts10])
eval10 = evaluate_all(y_true, y_pred)

print(f"Evaluation:\n{json.dumps(eval10, indent=4)}")

Evaluation     0 | Progress 0.00%
Evaluation   100 | Progress 8.70%
Evaluation   200 | Progress 17.39%
Evaluation   300 | Progress 26.09%
Evaluation   400 | Progress 34.78%
Evaluation   500 | Progress 43.48%
Evaluation   600 | Progress 52.17%
Evaluation   700 | Progress 60.87%
Evaluation   800 | Progress 69.57%
Evaluation   900 | Progress 78.26%
Evaluation  1000 | Progress 86.96%
Evaluation  1100 | Progress 95.65%
Evaluation:
{
    "roc-auc": 0.99,
    "brier": 0.915,
    "c@1": 0.949,
    "f1": 0.949,
    "f05u": 0.949,
    "mean": 0.95
}


In [9]:
classifier = TextClassification(
    device=None,
    embeddings_name='nlp_embeddings_20',
    classifier_name='nlp_classifier_20.pkl'
)

classified_texts20 = classifier.classify_texts(text_comparisons)
y_pred = np.array([text_comp["is_human"] for text_comp in classified_texts20])
eval20 = evaluate_all(y_true, y_pred)

print(f"Evaluation:\n{json.dumps(eval20, indent=4)}")

Evaluation     0 | Progress 0.00%
Evaluation   100 | Progress 8.70%
Evaluation   200 | Progress 17.39%
Evaluation   300 | Progress 26.09%
Evaluation   400 | Progress 34.78%
Evaluation   500 | Progress 43.48%
Evaluation   600 | Progress 52.17%
Evaluation   700 | Progress 60.87%
Evaluation   800 | Progress 69.57%
Evaluation   900 | Progress 78.26%
Evaluation  1000 | Progress 86.96%
Evaluation  1100 | Progress 95.65%
Evaluation:
{
    "roc-auc": 0.993,
    "brier": 0.925,
    "c@1": 0.948,
    "f1": 0.948,
    "f05u": 0.95,
    "mean": 0.953
}


In [10]:
import pandas as pd

evaluation = pd.DataFrame.from_records([eval3, eval10, eval20], index=["3-epoch-no-noise", "5-epoch-no-noise", "20-epoch-no-noise"])
evaluation

,roc-auc,brier,c@1,f1,f05u,mean
3-epoch-no-noise,0.986,0.906,0.933,0.933,0.937,0.939
5-epoch-no-noise,0.990,0.915,0.949,0.949,0.949,0.950
20-epoch-no-noise,0.993,0.925,0.948,0.948,0.950,0.953


```json
3-epoch no noise
{
    "roc-auc": 0.986,
    "brier": 0.906,
    "c@1": 0.933,
    "f1": 0.933,
    "f05u": 0.937,
    "mean": 0.939
}
10-epoch no noise
{
    "roc-auc": 0.99,
    "brier": 0.915,
    "c@1": 0.949,
    "f1": 0.949,
    "f05u": 0.949,
    "mean": 0.95
}
20-epoch no noise
{
    "roc-auc": 0.993,
    "brier": 0.925,
    "c@1": 0.948,
    "f1": 0.948,
    "f05u": 0.95,
    "mean": 0.953
}
```


Without noise, the scores obtained do not significantly improve over the previous ones, so we can deduce that the cases where our model fails were not significantly associated with noisy cases. In the following tests, we will increase the probability of word change and the amount of leetspeak introduced in noisy cases.


### Evaluation with More Noise than Training


We increase `change_prob` and `change_freq`, keeping the number of test cases the same for comparison with the results from the first evaluation.


In [ ]:
import os
import json

os.environ["TOKENIZERS_PARALLELISM"] = "false"
loader = DatasetLoader(
    storage_location="documents",
    storage_information={
        "tokenizer_name": "Alibaba-NLP/gte-base-en-v1.5",
        "dataset_types": {"train": 0.75, "validate": 0.15, "test": 0.1}
    }
)
generators = loader.create_generators(
    n_test_cases=1000,
    token_size=2,
    rep_noisy_cases=0.15,
    change_prob=0.25,
    change_freq=0.25
)
text_comparisons = generators["test"].generate_random_pairings()
y_true = np.array([text_comp["is_human"] for text_comp in text_comparisons])

In [7]:
classifier = TextClassification(
    device=None,
    embeddings_name='nlp_embeddings_3',
    classifier_name='nlp_classifier_3.pkl'
)

classified_texts3 = classifier.classify_texts(text_comparisons)
y_pred = np.array([text_comp["is_human"] for text_comp in classified_texts3])
eval3 = evaluate_all(y_true, y_pred)

print(f"Evaluation:\n{json.dumps(eval3, indent=4)}")

Evaluation     0 | Progress 0.00%
Evaluation   100 | Progress 8.70%
Evaluation   200 | Progress 17.39%
Evaluation   300 | Progress 26.09%
Evaluation   400 | Progress 34.78%
Evaluation   500 | Progress 43.48%
Evaluation   600 | Progress 52.17%
Evaluation   700 | Progress 60.87%
Evaluation   800 | Progress 69.57%
Evaluation   900 | Progress 78.26%
Evaluation  1000 | Progress 86.96%
Evaluation  1100 | Progress 95.65%
Evaluation:
{
    "roc-auc": 0.985,
    "brier": 0.905,
    "c@1": 0.932,
    "f1": 0.932,
    "f05u": 0.936,
    "mean": 0.938
}


In [8]:
classifier = TextClassification(
    device=None,
    embeddings_name='nlp_embeddings_10',
    classifier_name='nlp_classifier_10.pkl'
)

classified_texts10 = classifier.classify_texts(text_comparisons)
y_pred = np.array([text_comp["is_human"] for text_comp in classified_texts10])
eval10 = evaluate_all(y_true, y_pred)

print(f"Evaluation:\n{json.dumps(eval10, indent=4)}")

Evaluation     0 | Progress 0.00%
Evaluation   100 | Progress 8.70%
Evaluation   200 | Progress 17.39%
Evaluation   300 | Progress 26.09%
Evaluation   400 | Progress 34.78%
Evaluation   500 | Progress 43.48%
Evaluation   600 | Progress 52.17%
Evaluation   700 | Progress 60.87%
Evaluation   800 | Progress 69.57%
Evaluation   900 | Progress 78.26%
Evaluation  1000 | Progress 86.96%
Evaluation  1100 | Progress 95.65%
Evaluation:
{
    "roc-auc": 0.989,
    "brier": 0.913,
    "c@1": 0.942,
    "f1": 0.942,
    "f05u": 0.943,
    "mean": 0.946
}


In [9]:
classifier = TextClassification(
    device=None,
    embeddings_name='nlp_embeddings_20',
    classifier_name='nlp_classifier_20.pkl'
)

classified_texts20 = classifier.classify_texts(text_comparisons)
y_pred = np.array([text_comp["is_human"] for text_comp in classified_texts20])
eval20 = evaluate_all(y_true, y_pred)

print(f"Evaluation:\n{json.dumps(eval20, indent=4)}")

Evaluation     0 | Progress 0.00%
Evaluation   100 | Progress 8.70%
Evaluation   200 | Progress 17.39%
Evaluation   300 | Progress 26.09%
Evaluation   400 | Progress 34.78%
Evaluation   500 | Progress 43.48%
Evaluation   600 | Progress 52.17%
Evaluation   700 | Progress 60.87%
Evaluation   800 | Progress 69.57%
Evaluation   900 | Progress 78.26%
Evaluation  1000 | Progress 86.96%
Evaluation  1100 | Progress 95.65%
Evaluation:
{
    "roc-auc": 0.991,
    "brier": 0.922,
    "c@1": 0.945,
    "f1": 0.945,
    "f05u": 0.947,
    "mean": 0.95
}


In [10]:
import pandas as pd

evaluation = pd.DataFrame.from_records([eval3, eval10, eval20], index=["3-epoch-noisier", "5-epoch-noisier", "20-epoch-noisier"])
evaluation

,roc-auc,brier,c@1,f1,f05u,mean
3-epoch-noisier,0.985,0.905,0.932,0.932,0.936,0.938
5-epoch-noisier,0.989,0.913,0.942,0.942,0.943,0.946
20-epoch-noisier,0.991,0.922,0.945,0.945,0.947,0.950


Results obtained:
```json
3-epoch noisier
{
    "roc-auc": 0.985,
    "brier": 0.905,
    "c@1": 0.932,
    "f1": 0.932,
    "f05u": 0.936,
    "mean": 0.938
}
10-epoch noisier
{
    "roc-auc": 0.989,
    "brier": 0.913,
    "c@1": 0.942,
    "f1": 0.942,
    "f05u": 0.943,
    "mean": 0.946
}

20-epoch noisier
{
    "roc-auc": 0.991,
    "brier": 0.922,
    "c@1": 0.945,
    "f1": 0.945,
    "f05u": 0.947,
    "mean": 0.95
}
```


Even with the increased noise we introduced, the scores remain close to those of the first evaluation. It should be noted that if we had significantly increased the noise (20-30% increase in both word changes and per-word noise), we suspect the performance would have degraded.


### Evaluation with External Documents


Finally, to check how robust our model is against data not present in the competition datasets or our augmented data, we used a Kaggle dataset [AI_Human texts](https://www.kaggle.com/datasets/shanegerami/ai-vs-human-text/data) that includes 500,000 essays on a variety of topics (cars, US politics, etc.).

We also loaded them using our `DatasetLoader` class, and tested with 5,000 random pairs and about 750 of those with noise.


In [ ]:
import os
import pandas as pd
import json

os.environ["TOKENIZERS_PARALLELISM"] = "false"
loader = DatasetLoader(
    storage_location="documents_kaggle",
    storage_information={
        "tokenizer_name": "Alibaba-NLP/gte-base-en-v1.5",
        "dataset_types": {"test": 1.}
    }
)

# Path to your CSV file
csv_file_path = 'AI_Human.csv'

# Read the CSV file into a DataFrame
df = pd.read_csv(csv_file_path)

# Convert the DataFrame to a list of JSON objects (dictionaries)
texts = df.to_dict(orient='records')

loader.preprocess_files(texts)

In [5]:
generators = loader.create_generators(
    n_test_cases=5000,
    token_size=2,
    rep_noisy_cases=0.15,
    change_prob=0.25,
    change_freq=0.25
)
text_comparisons = generators["test"].generate_random_pairings()
y_true = np.array([text_comp["is_human"] for text_comp in text_comparisons])

In [6]:
classifier = TextClassification(
    device=None,
    embeddings_name='nlp_embeddings_3',
    classifier_name='nlp_classifier_3.pkl'
)

classified_texts3 = classifier.classify_texts(text_comparisons)
y_pred = np.array([text_comp["is_human"] for text_comp in classified_texts3])
eval3 = evaluate_all(y_true, y_pred)

print(f"Evaluation:\n{json.dumps(eval3, indent=4)}")

Evaluation     0 | Progress 0.00%
Evaluation   100 | Progress 1.74%
Evaluation   200 | Progress 3.48%
Evaluation   300 | Progress 5.22%
Evaluation   400 | Progress 6.96%
Evaluation   500 | Progress 8.70%
Evaluation   600 | Progress 10.43%
Evaluation   700 | Progress 12.17%
Evaluation   800 | Progress 13.91%
Evaluation   900 | Progress 15.65%
Evaluation  1000 | Progress 17.39%
Evaluation  1100 | Progress 19.13%
Evaluation  1200 | Progress 20.87%
Evaluation  1300 | Progress 22.61%
Evaluation  1400 | Progress 24.35%
Evaluation  1500 | Progress 26.09%
Evaluation  1600 | Progress 27.83%
Evaluation  1700 | Progress 29.57%
Evaluation  1800 | Progress 31.30%
Evaluation  1900 | Progress 33.04%
Evaluation  2000 | Progress 34.78%
Evaluation  2100 | Progress 36.52%
Evaluation  2200 | Progress 38.26%
Evaluation  2300 | Progress 40.00%
Evaluation  2400 | Progress 41.74%
Evaluation  2500 | Progress 43.48%
Evaluation  2600 | Progress 45.22%
Evaluation  2700 | Progress 46.96%
Evaluation  2800 | Progres

In [7]:
classifier = TextClassification(
    device=None,
    embeddings_name='nlp_embeddings_10',
    classifier_name='nlp_classifier_10.pkl'
)

classified_texts10 = classifier.classify_texts(text_comparisons)
y_pred = np.array([text_comp["is_human"] for text_comp in classified_texts10])
eval10 = evaluate_all(y_true, y_pred)

print(f"Evaluation:\n{json.dumps(eval10, indent=4)}")

Evaluation     0 | Progress 0.00%
Evaluation   100 | Progress 1.74%
Evaluation   200 | Progress 3.48%
Evaluation   300 | Progress 5.22%
Evaluation   400 | Progress 6.96%
Evaluation   500 | Progress 8.70%
Evaluation   600 | Progress 10.43%
Evaluation   700 | Progress 12.17%
Evaluation   800 | Progress 13.91%
Evaluation   900 | Progress 15.65%
Evaluation  1000 | Progress 17.39%
Evaluation  1100 | Progress 19.13%
Evaluation  1200 | Progress 20.87%
Evaluation  1300 | Progress 22.61%
Evaluation  1400 | Progress 24.35%
Evaluation  1500 | Progress 26.09%
Evaluation  1600 | Progress 27.83%
Evaluation  1700 | Progress 29.57%
Evaluation  1800 | Progress 31.30%
Evaluation  1900 | Progress 33.04%
Evaluation  2000 | Progress 34.78%
Evaluation  2100 | Progress 36.52%
Evaluation  2200 | Progress 38.26%
Evaluation  2300 | Progress 40.00%
Evaluation  2400 | Progress 41.74%
Evaluation  2500 | Progress 43.48%
Evaluation  2600 | Progress 45.22%
Evaluation  2700 | Progress 46.96%
Evaluation  2800 | Progres

In [8]:
classifier = TextClassification(
    device=None,
    embeddings_name='nlp_embeddings_20',
    classifier_name='nlp_classifier_20.pkl'
)

classified_texts20 = classifier.classify_texts(text_comparisons)
y_pred = np.array([text_comp["is_human"] for text_comp in classified_texts20])
eval20 = evaluate_all(y_true, y_pred)

print(f"Evaluation:\n{json.dumps(eval20, indent=4)}")

Evaluation     0 | Progress 0.00%
Evaluation   100 | Progress 1.74%
Evaluation   200 | Progress 3.48%
Evaluation   300 | Progress 5.22%
Evaluation   400 | Progress 6.96%
Evaluation   500 | Progress 8.70%
Evaluation   600 | Progress 10.43%
Evaluation   700 | Progress 12.17%
Evaluation   800 | Progress 13.91%
Evaluation   900 | Progress 15.65%
Evaluation  1000 | Progress 17.39%
Evaluation  1100 | Progress 19.13%
Evaluation  1200 | Progress 20.87%
Evaluation  1300 | Progress 22.61%
Evaluation  1400 | Progress 24.35%
Evaluation  1500 | Progress 26.09%
Evaluation  1600 | Progress 27.83%
Evaluation  1700 | Progress 29.57%
Evaluation  1800 | Progress 31.30%
Evaluation  1900 | Progress 33.04%
Evaluation  2000 | Progress 34.78%
Evaluation  2100 | Progress 36.52%
Evaluation  2200 | Progress 38.26%
Evaluation  2300 | Progress 40.00%
Evaluation  2400 | Progress 41.74%
Evaluation  2500 | Progress 43.48%
Evaluation  2600 | Progress 45.22%
Evaluation  2700 | Progress 46.96%
Evaluation  2800 | Progres

In [9]:
import pandas as pd

evaluation = pd.DataFrame.from_records([eval3, eval10, eval20], index=["3-epoch-kaggle", "5-epoch-kaggle", "20-epoch-kaggle"])
evaluation

,roc-auc,brier,c@1,f1,f05u,mean
3-epoch-kaggle,0.938,0.863,0.861,0.860,0.863,0.877
5-epoch-kaggle,0.945,0.870,0.878,0.878,0.879,0.890
20-epoch-kaggle,0.948,0.872,0.878,0.878,0.877,0.891


```json
3-epoch kaggle
{
    "roc-auc": 0.938,
    "brier": 0.863,
    "c@1": 0.861,
    "f1": 0.86,
    "f05u": 0.863,
    "mean": 0.877
}
10-epoch kaggle
{
    "roc-auc": 0.945,
    "brier": 0.87,
    "c@1": 0.878,
    "f1": 0.878,
    "f05u": 0.879,
    "mean": 0.89
}
20-epoch kaggle
{
    "roc-auc": 0.948,
    "brier": 0.872,
    "c@1": 0.878,
    "f1": 0.878,
    "f05u": 0.877,
    "mean": 0.891
}
```


These results more closely resemble those obtained in the competition, with our best model performing above the third-best competition model, and with an ROC-AUC very close to the first-place model.

This indicates that our model could be improved by, for example, introducing the Kaggle data into training to increase data diversity, finding human and AI data sources in multiple languages and adding them to training, or using a more powerful multimodal embedding model.

If we had to submit a model to the competition, we would select the 20-epoch model.
